In [1]:
!nvcc --version
!pip install git+https://github.com/afnan47/cuda.git
%load_ext nvcc_plugin

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-n9rcmi_i
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-n9rcmi_i
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-py3-none-any.whl size=4290 sha256=4e9908e68b388f0e102e73a0e012b145bb165d50d7ee86bee13b67376ed52393
  Stored in directory: /tmp/pip-ephem-wheel-cache-kc6a439p/wheels/e8/cf/c3/c90ca0d0bba7969f9b8670f5624f76d097123d656355c77053
Successfully built NVCCPlugin
created output directory at /content/src
Out bin /content/result.out


In [3]:
%%cu
#include <iostream>
#include <cuda.h>

using namespace std;

#define BLOCK_SIZE 2

/* -------- CUDA KERNEL -------- */
__global__ void gpuMM(float *A, float *B, float *C, int N) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    float sum = 0.0f;

    for (int n = 0; n < N; ++n) {
        sum += A[row * N + n] * B[n * N + col];
    }

    C[row * N + col] = sum;
}

/* -------- MAIN FUNCTION -------- */
int main() {

    int N;
    float K;

    cout << "Enter a value for Size/2 of matrix: ";
    cin >> K;

    K = 1;   // fixed for demo
    N = K * BLOCK_SIZE;

    cout << "\nExecuting Matrix Multiplication\n";
    cout << "Matrix size: " << N << " x " << N << endl;

    /* -------- HOST MEMORY -------- */
    float *hA, *hB, *hC;

    hA = new float[N * N];
    hB = new float[N * N];
    hC = new float[N * N];

    /* -------- INITIALIZE MATRICES -------- */
    for (int i = 0; i < N * N; i++) {
        hA[i] = 2;
        hB[i] = 4;
    }

    /* -------- DEVICE MEMORY -------- */
    float *dA, *dB, *dC;
    int size = N * N * sizeof(float);

    cudaMalloc(&dA, size);
    cudaMalloc(&dB, size);
    cudaMalloc(&dC, size);

    /* -------- COPY DATA TO GPU -------- */
    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, size, cudaMemcpyHostToDevice);

    /* -------- THREADS & BLOCKS -------- */
    dim3 threadBlock(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid(K, K);

    /* -------- PRINT INPUT MATRICES -------- */
    cout << "\nMatrix A:\n";
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            cout << hA[row * N + col] << " ";
        }
        cout << endl;
    }

    cout << "\nMatrix B:\n";
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            cout << hB[row * N + col] << " ";
        }
        cout << endl;
    }

    /* -------- KERNEL CALL -------- */
    gpuMM<<<grid, threadBlock>>>(dA, dB, dC, N);

    /* -------- COPY RESULT BACK -------- */
    float *C = new float[N * N];
    cudaMemcpy(C, dC, size, cudaMemcpyDeviceToHost);

    /* -------- PRINT RESULT -------- */
    cout << "\nResultant Matrix:\n";
    for (int row = 0; row < N; row++) {
        for (int col = 0; col < N; col++) {
            cout << C[row * N + col] << " ";
        }
        cout << endl;
    }

    cout << "\nFinished.\n";

    /* -------- FREE MEMORY -------- */
    delete[] hA;
    delete[] hB;
    delete[] hC;
    delete[] C;

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);

    return 0;
}

Enter a value for Size/2 of matrix: 
Executing Matrix Multiplication
Matrix size: 2 x 2

Matrix A:
2 2 
2 2 

Matrix B:
4 4 
4 4 

Resultant Matrix:
16 16 
16 16 

Finished.

